### conversation.py

In [ ]:
"""
Conversation State Manager for DataForge
Handles generation IDs, cancellation fencing, and heard-context tracking.
"""

import asyncio
from typing import List, Dict, Any, Optional


class ConversationState:
    """
    Manages per-connection conversation state with generation-based fencing.
    
    Key concepts:
    - Each user query gets a monotonically increasing generationId
    - Interruptions mark a generation as cancelled via asyncio.Event
    - heard_context tracks what the user actually heard (for follow-ups)
    - Stale results are silently discarded if their generation is cancelled
    """

    def __init__(self):
        self.messages: List[Dict[str, Any]] = []
        self.current_generation_id: int = 0
        self.active_tasks: Dict[int, asyncio.Event] = {}
        self.heard_context: List[Dict[str, Any]] = []
        self.last_query_plan: dict | None = None

    def new_query(self, text: str) -> int:
        """
        Register a new user query.
        Returns the assigned generationId.
        """
        self.current_generation_id += 1
        gen_id = self.current_generation_id

        msg = {
            "role": "user",
            "content": text,
            "generationId": gen_id,
            "wasHeard": True,
        }
        self.messages.append(msg)
        self.heard_context.append(msg)

        # Create cancellation event (set = cancelled)
        self.active_tasks[gen_id] = asyncio.Event()
        return gen_id

    def interrupt(self, generation_id: int):
        """
        Mark a generation as cancelled.
        Sets the cancellation event so background tasks can check it.
        """
        if generation_id in self.active_tasks:
            self.active_tasks[generation_id].set()

        # Mark any assistant response for this generation as not heard
        for msg in self.messages:
            if (
                msg.get("generationId") == generation_id
                and msg.get("role") == "assistant"
            ):
                msg["wasHeard"] = False
                if msg in self.heard_context:
                    self.heard_context.remove(msg)

    def add_response(
        self, text: str, generation_id: int, was_heard: bool = True
    ):
        """Record an assistant response."""
        msg = {
            "role": "assistant",
            "content": text,
            "generationId": generation_id,
            "wasHeard": was_heard,
        }
        self.messages.append(msg)
        if was_heard:
            self.heard_context.append(msg)

    def is_cancelled(self, generation_id: int) -> bool:
        """Check if a generation has been cancelled."""
        evt = self.active_tasks.get(generation_id)
        if evt is None:
            return True  # Unknown generation = treat as cancelled
        return evt.is_set()

    def store_query_plan(self, plan: dict):
        """Store the last executed query plan for context in follow-ups."""
        self.last_query_plan = plan

    def get_context_for_llm(self) -> Dict[str, Any]:
        """
        Return conversation context based on what the user actually heard.
        This ensures follow-up queries reference the correct state.
        """
        messages = [
            {"role": m["role"], "content": m["content"]}
            for m in self.heard_context[-10:]  # Last 10 messages
        ]
        return {
            "messages": messages,
            "last_query_plan": self.last_query_plan
        }

    def mark_heard(self, generation_id: int):
        """Mark all messages for a generation as heard."""
        for msg in self.messages:
            if msg.get("generationId") == generation_id:
                msg["wasHeard"] = True
                if msg not in self.heard_context:
                    self.heard_context.append(msg)

    def cleanup_old_generations(self, keep_last: int = 20):
        """Remove old cancellation events to prevent memory leaks."""
        if len(self.active_tasks) > keep_last:
            old_ids = sorted(self.active_tasks.keys())[:-keep_last]
            for old_id in old_ids:
                del self.active_tasks[old_id]


### data_engine.py

In [ ]:
"""
Data Analysis Engine for DataForge
Generates synthetic datasets and executes Pandas operations.
"""

import pandas as pd
import numpy as np
import asyncio
import logging
from typing import Dict, Any, List, Optional

logger = logging.getLogger("dataforge.data")


class DataEngine:
    """
    In-memory data analysis engine with three sample datasets.
    Uses fixed random seed for reproducibility.
    """

    def __init__(self):
        self.datasets: Dict[str, pd.DataFrame] = {}
        self.dataset_descriptions: Dict[str, str] = {}
        self._generate_synthetic_data()
        logger.info(f"DataEngine initialized with {len(self.datasets)} datasets")

    def _generate_synthetic_data(self):
        """Generate realistic synthetic datasets with fixed seed."""
        np.random.seed(42)

        # ──────────────────────────────────────────────────────
        # 1. Sales Data — 200 transactions across regions/products/quarters
        # ──────────────────────────────────────────────────────
        regions = ["North", "South", "East", "West"]
        products = ["Widget Alpha", "Widget Beta", "Gadget Pro", "Gadget Lite", "Service Plus"]
        quarters = ["Q1", "Q2", "Q3", "Q4"]
        sales_reps = ["Alice", "Bob", "Carol", "David", "Eve", "Frank"]

        sales_rows = []
        for i in range(200):
            region = np.random.choice(regions)
            product = np.random.choice(products)
            quarter = np.random.choice(quarters)
            # Make sales somewhat realistic — different products have different price ranges
            base_price = {"Widget Alpha": 5000, "Widget Beta": 8000, "Gadget Pro": 15000,
                          "Gadget Lite": 3000, "Service Plus": 12000}
            amount = np.random.normal(base_price[product], base_price[product] * 0.3)
            amount = max(500, round(amount, 2))
            sales_rows.append({
                "region": region,
                "product": product,
                "quarter": quarter,
                "sales_rep": np.random.choice(sales_reps),
                "amount": amount,
                "units": np.random.randint(1, 50),
            })
        self.datasets["sales"] = pd.DataFrame(sales_rows)

        # ──────────────────────────────────────────────────────
        # 2. User Analytics — 365 days of daily metrics
        # ──────────────────────────────────────────────────────
        dates = pd.date_range(start="2024-01-01", periods=365)
        base_dau = 8000
        # Add growth trend + weekly seasonality
        trend = np.linspace(0, 3000, 365)
        weekly = 1500 * np.sin(np.arange(365) * 2 * np.pi / 7)
        noise = np.random.normal(0, 500, 365)
        dau = (base_dau + trend + weekly + noise).astype(int).clip(min=2000)

        sessions = (dau * np.random.uniform(1.2, 1.8, 365)).astype(int)
        bounce_rate = np.clip(np.random.normal(0.45, 0.08, 365), 0.15, 0.85)
        avg_duration = np.clip(np.random.normal(4.5, 1.2, 365), 1.0, 12.0)

        self.datasets["users"] = pd.DataFrame({
            "date": dates.strftime("%Y-%m-%d"),
            "daily_active_users": dau,
            "sessions": sessions,
            "bounce_rate": bounce_rate.round(3),
            "avg_session_duration_min": avg_duration.round(1),
            "new_users": (dau * np.random.uniform(0.05, 0.15, 365)).astype(int),
            "page_views": (sessions * np.random.uniform(3, 8, 365)).astype(int),
        })

        # ──────────────────────────────────────────────────────
        # 3. Financial Data — 60 months of revenue/expenses
        # ──────────────────────────────────────────────────────
        months = pd.date_range(start="2020-01-01", periods=60, freq="MS")
        categories = ["Software", "Services", "Hardware"]
        
        fin_rows = []
        for i, month in enumerate(months):
            cat = categories[i % 3]
            # Revenue grows over time
            base_rev = 80000 + i * 1500
            revenue = np.random.normal(base_rev, base_rev * 0.15)
            expenses = np.random.normal(revenue * 0.65, revenue * 0.1)
            fin_rows.append({
                "month": month.strftime("%Y-%m"),
                "revenue": round(max(20000, revenue), 2),
                "expenses": round(max(15000, expenses), 2),
                "profit": round(revenue - expenses, 2),
                "category": cat,
                "headcount": int(30 + i * 0.8 + np.random.randint(-2, 3)),
            })
        self.datasets["financials"] = pd.DataFrame(fin_rows)
        
        self.dataset_descriptions["sales"] = "Sales transactions by region, product, quarter, and rep"
        self.dataset_descriptions["users"] = "Daily user analytics: DAU, sessions, bounce rate, page views"
        self.dataset_descriptions["financials"] = "Monthly financial data: revenue, expenses, profit by category"

    def get_dataset_profile(self, dataset_name: str) -> dict:
        """Return detailed profile of a dataset."""
        df = self.datasets.get(dataset_name)
        if df is None:
            return {}
        
        columns_info = []
        for col in df.columns:
            dtype_str = "text"
            if pd.api.types.is_numeric_dtype(df[col]):
                dtype_str = "numeric"
            elif pd.api.types.is_datetime64_any_dtype(df[col]):
                dtype_str = "datetime"
            elif pd.api.types.is_categorical_dtype(df[col]) or df[col].nunique() < 20:
                dtype_str = "categorical"

            unique_vals = df[col].dropna().unique()
            col_info = {
                "name": col,
                "dtype": dtype_str,
                "unique_count": len(unique_vals),
                "sample_values": [str(v) for v in unique_vals[:5]]
            }
            if dtype_str == "numeric":
                col_info["min"] = float(df[col].min()) if not pd.isna(df[col].min()) else None
                col_info["max"] = float(df[col].max()) if not pd.isna(df[col].max()) else None
                col_info["mean"] = float(df[col].mean()) if not pd.isna(df[col].mean()) else None
            
            columns_info.append(col_info)

        return {
            "name": dataset_name,
            "description": self.dataset_descriptions.get(dataset_name, "Uploaded dataset"),
            "rows": len(df),
            "columns": columns_info,
            "sample_rows": df.head(3).fillna("").to_dict(orient="records")
        }

    def list_datasets(self) -> List[Dict[str, Any]]:
        """Return metadata about all available datasets."""
        metadata = []
        for name in self.datasets.keys():
            metadata.append(self.get_dataset_profile(name))
        return metadata

    def add_dataset(self, name: str, df: pd.DataFrame, description: str = "Uploaded dataset"):
        """Add a new dataset to the engine."""
        self.datasets[name] = df
        self.dataset_descriptions[name] = description
        logger.info(f"Added new dataset '{name}' with {len(df)} rows")

    async def execute_query(
        self,
        query_plan: Dict[str, Any],
        cancel_event: Optional[asyncio.Event] = None,
    ) -> Dict[str, Any]:
        """
        Execute an LLM-generated query plan against the datasets.
        Checks cancel_event between operations to support interruption.
        """
        dataset_name = query_plan.get("dataset", "")
        operations = query_plan.get("operations", [])

        if dataset_name not in self.datasets:
            logger.warning(f"Dataset '{dataset_name}' not found, falling back to 'sales'")
            dataset_name = "sales"

        df = self.datasets[dataset_name].copy()

        for op in operations:
            # Check for cancellation between operations
            if cancel_event and cancel_event.is_set():
                return {"data": [], "columns": [], "cancelled": True}

            op_type = op.get("type", "")
            params = op.get("params", {})

            try:
                if op_type == "filter":
                    col = params.get("column", "")
                    val = params.get("value")
                    operator = params.get("operator", "==")
                    if col in df.columns and val is not None:
                        if operator == "==":
                            df = df[df[col] == val]
                        elif operator == ">":
                            df = df[df[col] > float(val)]
                        elif operator == "<":
                            df = df[df[col] < float(val)]
                        elif operator == ">=":
                            df = df[df[col] >= float(val)]
                        elif operator == "<=":
                            df = df[df[col] <= float(val)]
                        elif operator == "!=":
                            df = df[df[col] != val]
                        elif operator == "contains":
                            df = df[df[col].astype(str).str.contains(str(val), case=False)]
                        elif operator == "in":
                            df = df[df[col].isin(val if isinstance(val, list) else [val])]
                        elif operator == "not_in":
                            df = df[~df[col].isin(val if isinstance(val, list) else [val])]

                elif op_type == "groupby_agg":
                    group_col = params.get("group_col", "")
                    agg_col = params.get("agg_col", "")
                    agg_func = params.get("agg_func", "sum")
                    if group_col in df.columns and agg_col in df.columns:
                        df = df.groupby(group_col, as_index=False)[agg_col].agg(agg_func)

                elif op_type == "sort":
                    col = params.get("column", "")
                    ascending = params.get("ascending", False)
                    if col in df.columns:
                        df = df.sort_values(by=col, ascending=ascending)

                elif op_type == "top_n":
                    col = params.get("column", "")
                    n = params.get("n", 10)
                    ascending = params.get("ascending", False)
                    if col in df.columns:
                        df = df.sort_values(by=col, ascending=ascending).head(n)

                elif op_type == "value_counts":
                    col = params.get("column", "")
                    if col in df.columns:
                        df = df[col].value_counts().reset_index()
                        df.columns = [col, 'count']

                elif op_type == "describe":
                    df = df.describe().reset_index()

                elif op_type == "date_filter":
                    col = params.get("column", "")
                    start = params.get("start")
                    end = params.get("end")
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col])
                        if start:
                            df = df[df[col] >= pd.to_datetime(start)]
                        if end:
                            df = df[df[col] <= pd.to_datetime(end)]
                        df[col] = df[col].dt.strftime('%Y-%m-%d')

                elif op_type == "multi_group":
                    group_cols = params.get("group_cols", [])
                    agg_col = params.get("agg_col", "")
                    agg_func = params.get("agg_func", "sum")
                    valid_cols = [c for c in group_cols if c in df.columns]
                    if valid_cols and agg_col in df.columns:
                        df = df.groupby(valid_cols, as_index=False)[agg_col].agg(agg_func)

                elif op_type == "rename":
                    mapping = params.get("mapping", {})
                    df = df.rename(columns=mapping)

            except Exception as e:
                logger.warning(f"Operation {op_type} failed: {e}")
                continue

            # Small yield to event loop
            await asyncio.sleep(0)

        # Handle NaN for JSON serialization
        df = df.fillna(0)

        # Round numeric columns for cleaner display
        for col in df.select_dtypes(include=[np.number]).columns:
            df[col] = df[col].round(2)

        records = df.head(100).to_dict(orient="records")  # Cap at 100 rows for charts
        return {"data": records, "columns": list(df.columns)}

    def get_chart_data(
        self,
        data: List[Dict],
        chart_type: str,
        x_col: str,
        y_col: str,
        title: str,
    ) -> Dict[str, Any]:
        """Format data for Recharts frontend consumption."""
        return {
            "chartType": chart_type,
            "data": data,
            "config": {"x": x_col, "y": y_col, "title": title},
        }


### llm_service.py

In [ ]:
"""
LLM Service for DataForge — Google Gemini Integration
Translates natural language queries into data analysis plans.
"""

import json
import os
import re
import logging
from typing import Dict, Any, List

from google import genai
from google.genai import types

logger = logging.getLogger("dataforge.llm")

SYSTEM_PROMPT = """You are DataForge, a highly intelligent voice-native AI data analyst. You analyze ANY dataset — built-in or user-uploaded — and produce the most insightful analysis possible.
    
## Available Datasets
{datasets}

## Response Types — Choose the BEST representation:
- "chart": Data that has clear visual patterns (trends, comparisons, distributions)
- "table": Raw records, filtered lists, or data that needs exact values shown
- "chart_and_insight": Chart WITH detailed analytical commentary (PREFERRED for most queries)
- "insight": Pure text analysis when no visualization makes sense (e.g., "what's the average?")

## Chart Types — Pick the MOST appropriate:
- "bar": Comparing categories (sales by region, counts by type)
- "line": Trends over time (daily users, monthly revenue)
- "area": Cumulative trends, volume over time
- "pie": Proportional breakdown (< 8 categories only)
- "scatter": Correlation between two numeric variables
- "stacked_bar": Category comparison with sub-breakdowns
- "horizontal_bar": When category labels are long
- "composed": Overlay bar + line (e.g., revenue bars + profit line)

## Multi-turn Follow-ups (CRITICAL)
You are in a conversation. Check the `Previous query plan` section carefully.
- If the user says "filter that by X", "only show Y", "break it down by Z", "what about Q1", "now show me...", "for the North region only" — this is a FOLLOW-UP.
- For follow-ups: use the SAME dataset, KEEP existing operations, and ADD/MODIFY the relevant filter or grouping.
- Previous query plan will show you exactly what dataset and operations were used last.

## Rules for spoken_response (read aloud by TTS):
1. Max 2-3 short conversational sentences
2. Round large numbers: say "about 2.5 million" not "2,487,321"
3. No bullet points, lists, or markdown
4. Lead with the key insight

## Rules for detailed_insights (shown as text in the UI):
- Provide 3-6 bullet points of analytical findings
- Include specific numbers and percentages
- Note outliers, trends, patterns, and anomalies
- Compare categories or time periods when relevant
- Suggest possible explanations or next questions
- Use markdown formatting (bold for emphasis, bullet points)

## Rules for operations:
- Use EXACT column names from the dataset
- Operation types: filter, groupby_agg, sort, top_n, value_counts, date_filter, multi_group, rename
- filter: {{"type":"filter","params":{{"column":"col","value":"val","operator":"=="}}}}
- groupby_agg: {{"type":"groupby_agg","params":{{"group_col":"col","agg_col":"col2","agg_func":"sum"}}}}
- sort: {{"type":"sort","params":{{"column":"col","ascending":false}}}}
- top_n: {{"type":"top_n","params":{{"column":"col","n":10,"ascending":false}}}}
- value_counts: {{"type":"value_counts","params":{{"column":"col"}}}}
- date_filter: {{"type":"date_filter","params":{{"column":"date","start":"2024-01-01","end":"2024-06-30"}}}}
- multi_group: {{"type":"multi_group","params":{{"group_cols":["col1","col2"],"agg_col":"val","agg_func":"sum"}}}}
- Operators: ==, >, <, >=, <=, !=, contains, in, not_in (for 'in' and 'not_in', 'value' MUST be an array of strings/numbers)
- Aggregation functions: sum, mean, count, min, max

## You MUST return ONLY a JSON object with these fields:
- dataset: string (dataset ID)
- operations: array of operation objects
- response_type: "chart" | "table" | "insight" | "chart_and_insight"
- chart_type: string (one of the chart types above, or null if response_type is "insight" or "table")
- chart_config: {{"x": "column", "y": "column", "title": "Chart Title"}} (or null)
- spoken_response: string (what to say aloud, 2-3 sentences max)
- detailed_insights: string (markdown-formatted analytical findings, 3-6 bullet points)
- filler_phrase: string (short phrase like "Let me analyze that")
"""


class LLMService:
    """Google Gemini LLM service."""

    def __init__(self, api_key: str = None):
        self.api_key = api_key or os.getenv("GEMINI_API_KEY", "")
        self.client = genai.Client(api_key=self.api_key)

    async def analyze_query(
        self,
        user_query: str,
        context: Dict[str, Any],
        available_datasets: List[Dict[str, Any]],
    ) -> Dict[str, Any]:
        """Analyze a user query and produce a structured analysis plan."""
        datasets_str = json.dumps(available_datasets, indent=2)
        system = SYSTEM_PROMPT.format(datasets=datasets_str)

        # Build context
        context_str = ""
        if context:
            messages = context.get("messages", [])
            if messages:
                context_str = "\nRecent conversation:\n"
                for msg in messages[-4:]:
                    role = msg.get("role", "user")
                    content = msg.get("content", "")
                    context_str += f"- {role}: {content}\n"
            
            last_plan = context.get("last_query_plan")
            if last_plan:
                context_str += f"\nPrevious query plan:\n{json.dumps(last_plan, indent=2)}\n"

        user_prompt = f"{context_str}\nUser query: {user_query}\n\nReturn ONLY a valid JSON object, nothing else."

        try:
            response = await self.client.aio.models.generate_content(
                model="gemini-3.6-flash",
                contents=[
                    types.Content(role="user", parts=[
                        types.Part.from_text(text=system + "\n\n" + user_prompt)
                    ])
                ],
                config=types.GenerateContentConfig(
                    temperature=0.3,
                    max_output_tokens=1024,
                ),
            )

            raw_text = response.text.strip()
            logger.info(f"LLM raw response: {raw_text[:200]}")

            # Extract JSON from response (handle markdown code blocks)
            result = self._extract_json(raw_text)

            # Validate required fields
            required = ["dataset", "spoken_response"]
            for field in required:
                if field not in result:
                    raise ValueError(f"Missing required field: {field}")

            # Add defaults for optional/new fields
            result["response_type"] = result.get("response_type", "chart_and_insight")
            result["detailed_insights"] = result.get("detailed_insights", "")
            if "operations" not in result:
                result["operations"] = []
                
            return result

        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "quota" in error_str:
                logger.error("Quota error detected.")
                return {
                    "dataset": "sales",
                    "operations": [],
                    "response_type": "insight",
                    "detailed_insights": "The AI service is currently experiencing high demand or has exceeded its quota limit. Please try again later.",
                    "chart_type": None,
                    "chart_config": None,
                    "spoken_response": "I'm sorry, but I've reached my quota limit for now. Please try again later.",
                    "filler_phrase": "Let me check."
                }
            logger.error(f"LLM analysis failed: {e}", exc_info=True)
            return self._fallback_response(user_query)

    def _extract_json(self, text: str) -> dict:
        """Extract JSON from LLM response, handling markdown code blocks."""
        # Try direct parse first
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Try extracting from ```json ... ``` blocks
        json_match = re.search(r'```(?:json)?\s*\n?(.*?)\n?\s*```', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1).strip())
            except json.JSONDecodeError:
                pass

        # Try finding first { ... } block
        brace_match = re.search(r'\{.*\}', text, re.DOTALL)
        if brace_match:
            try:
                return json.loads(brace_match.group(0))
            except json.JSONDecodeError:
                pass

        raise ValueError(f"Could not extract JSON from LLM response: {text[:100]}")

    def _fallback_response(self, query: str) -> Dict[str, Any]:
        """Produce a safe fallback response when LLM fails."""
        query_lower = query.lower()

        if any(w in query_lower for w in ["sale", "revenue", "region", "product", "quarter"]):
            return {
                "dataset": "sales",
                "operations": [{"type": "groupby_agg", "params": {"group_col": "region", "agg_col": "amount", "agg_func": "sum"}}],
                "response_type": "chart_and_insight",
                "detailed_insights": "- Strong performance across regions.\n- Consider focusing on underperforming areas.",
                "chart_type": "bar",
                "chart_config": {"x": "region", "y": "amount", "title": "Sales by Region"},
                "spoken_response": "Here's a breakdown of sales across all regions. The chart shows the total sales amount for each region.",
                "filler_phrase": "Pulling up the sales data.",
            }
        elif any(w in query_lower for w in ["user", "growth", "active", "session", "traffic"]):
            return {
                "dataset": "users",
                "operations": [],
                "response_type": "chart_and_insight",
                "detailed_insights": "- Active users track consistently.\n- Some seasonal trends observed.",
                "chart_type": "line",
                "chart_config": {"x": "date", "y": "daily_active_users", "title": "Daily Active Users"},
                "spoken_response": "Here's the daily active users trend over time. You can see the overall growth pattern in the chart.",
                "filler_phrase": "Checking the user analytics.",
            }
        elif any(w in query_lower for w in ["financ", "profit", "expense", "revenue", "money", "cost"]):
            return {
                "dataset": "financials",
                "operations": [],
                "response_type": "chart_and_insight",
                "detailed_insights": "- Revenue shows steady growth.\n- Expenses remain proportional.",
                "chart_type": "area",
                "chart_config": {"x": "month", "y": "revenue", "title": "Monthly Revenue"},
                "spoken_response": "Here's the monthly revenue overview. The chart shows the revenue trend over the reporting period.",
                "filler_phrase": "Looking at the financials.",
            }
        else:
            return {
                "dataset": "sales",
                "operations": [{"type": "groupby_agg", "params": {"group_col": "region", "agg_col": "amount", "agg_func": "sum"}}],
                "response_type": "chart_and_insight",
                "detailed_insights": "- Data suggests regional variances.\n- Further breakdowns might be useful.",
                "chart_type": "bar",
                "chart_config": {"x": "region", "y": "amount", "title": "Sales Overview"},
                "spoken_response": "Here's a general overview of the sales data broken down by region.",
                "filler_phrase": "Let me look into that.",
            }


### rime_tts.py

In [ ]:
"""
Rime TTS Client for DataForge
Handles synthesis, filler speech, and cancellation.

Rime Configuration:
  - Model: coda (flagship)
  - Speaker: celeste
  - Language: en
  - Endpoint: https://users.rime.ai/v1/rime-tts
  - Audio Format: mp3 (Accept: audio/mpeg)
  - Transport: HTTP (full response, then chunk for streaming)
"""

import httpx
import asyncio
import os
import base64
import logging
import random
from typing import AsyncGenerator, Optional

logger = logging.getLogger("dataforge.rime")

# Contextual filler phrases categorized by query type
FILLER_PHRASES = {
    "sales": [
        "Let me pull up the sales figures.",
        "Analyzing the sales data now.",
        "Crunching those sales numbers for you.",
    ],
    "users": [
        "Let me check the user analytics.",
        "Pulling up the user data now.",
        "Looking into the user metrics.",
    ],
    "financials": [
        "Let me review the financial data.",
        "Analyzing the financial records now.",
        "Crunching the revenue numbers.",
    ],
    "default": [
        "Let me analyze that for you.",
        "Working on that right now.",
        "Let me crunch those numbers.",
        "Pulling up the data now.",
        "Analyzing the information.",
        "Give me just a moment.",
    ],
}


def pick_filler(query_text: str) -> str:
    """Select a contextual filler phrase based on the query content."""
    query_lower = query_text.lower()
    if any(w in query_lower for w in ["sale", "revenue", "product", "region"]):
        phrases = FILLER_PHRASES["sales"]
    elif any(w in query_lower for w in ["user", "session", "bounce", "active"]):
        phrases = FILLER_PHRASES["users"]
    elif any(w in query_lower for w in ["financ", "profit", "expense", "cost"]):
        phrases = FILLER_PHRASES["financials"]
    else:
        phrases = FILLER_PHRASES["default"]
    return random.choice(phrases)


class RimeTTS:
    """
    Rime TTS client with cancellation support.
    
    Uses the Coda model with the celeste voice.
    Synthesizes full audio, then sends in properly-sized chunks
    to avoid broken MP3 frames in the browser.
    """

    RIME_ENDPOINT = "https://users.rime.ai/v1/rime-tts"
    MODEL_ID = "coda"
    SPEAKER = "celeste"

    def __init__(self, api_key: str = None, region: str = None):
        self.api_key = api_key or os.getenv("RIME_API_KEY", "")
        # Route requests to the Rime region closest to application for lowest latency
        # See: https://docs.rime.ai/docs/regional-endpoints
        self.region = region or os.getenv("RIME_REGION", "west").lower()
        if self.region == "east":
            self.endpoint = "https://users-east.rime.ai/v1/rime-tts"
        else:
            self.endpoint = "https://users.rime.ai/v1/rime-tts" # Default (us-west-2)
            
        self.active_generation_id: Optional[int] = None
        self.last_filler_text: str = ""
        self._client: Optional[httpx.AsyncClient] = None

    async def _get_client(self) -> httpx.AsyncClient:
        """Reuse httpx client for connection pooling."""
        if self._client is None or self._client.is_closed:
            self._client = httpx.AsyncClient(timeout=30.0)
        return self._client

    def _headers(self) -> dict:
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "Accept": "audio/mpeg",
        }

    def _body(self, text: str, speed: float = 1.0) -> dict:
        return {
            "text": text,
            "speaker": self.SPEAKER,
            "modelId": self.MODEL_ID,
            "speedAlpha": speed,
            "reduceLatency": True,
        }

    def cancel(self):
        """Cancel the current synthesis by invalidating generation ID."""
        self.active_generation_id = None

    async def synthesize_filler(self, query_text: str) -> Optional[str]:
        """
        Synthesize a short contextual filler phrase.
        Returns base64-encoded MP3 audio string, or None on failure.
        """
        filler_text = pick_filler(query_text)
        self.last_filler_text = filler_text

        if not self.api_key:
            logger.warning("No Rime API key — skipping filler")
            return None

        try:
            client = await self._get_client()
            response = await client.post(
                self.endpoint,
                headers=self._headers(),
                json=self._body(filler_text, speed=1.1),
            )
            response.raise_for_status()
            return base64.b64encode(response.content).decode("utf-8")
        except Exception as e:
            logger.error(f"Filler synthesis failed: {e}")
            return None

    async def synthesize_streaming(
        self, text: str, generation_id: int
    ) -> AsyncGenerator[str, None]:
        """
        Synthesize full audio then yield in large chunks for smooth playback.
        
        Instead of streaming tiny 4KB chunks (which cause broken MP3 frames
        and crackling in the browser), we fetch the complete audio and split
        it into properly-sized chunks that the browser can decode cleanly.
        """
        self.active_generation_id = generation_id

        if not self.api_key:
            logger.warning("No Rime API key — yielding nothing")
            return

        try:
            # Fetch complete audio (Rime is fast enough for <3 sentence responses)
            client = await self._get_client()
            response = await client.post(
                self.RIME_ENDPOINT,
                headers=self._headers(),
                json=self._body(text),
                timeout=20.0,
            )
            response.raise_for_status()
            
            audio_data = response.content
            
            if self.active_generation_id != generation_id:
                return
            
            # Send as a single complete audio chunk for clean playback
            # This avoids MP3 frame boundary issues entirely
            if audio_data:
                yield base64.b64encode(audio_data).decode("utf-8")
                
        except httpx.HTTPStatusError as e:
            logger.error(f"Rime API error {e.response.status_code}: {e.response.text[:200]}")
        except Exception as e:
            logger.error(f"TTS synthesis error: {e}")

    async def close(self):
        """Close the HTTP client."""
        if self._client and not self._client.is_closed:
            await self._client.aclose()


### main.py

In [ ]:
"""
DataForge Backend — FastAPI WebSocket Server
Voice-native real-time data analysis with Rime TTS
"""

import os
import json
import asyncio
import logging
import time
from pathlib import Path
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
import pandas as pd
import io
from collections import deque

from conversation import ConversationState
from data_engine import DataEngine
from rime_tts import RimeTTS
from llm_service import LLMService

# Load .env for local development (Railway injects env vars directly)
try:
    env_path = Path(__file__).resolve().parent.parent / ".env"
    if env_path.exists():
        load_dotenv(env_path)
    local_env = Path(__file__).resolve().parent / ".env"
    if local_env.exists():
        load_dotenv(local_env)
except Exception:
    pass  # On Railway/Render, env vars are injected by the platform

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("dataforge")

app = FastAPI(
    title="DataForge API",
    description="Voice-native real-time data analyst powered by Rime TTS",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Shared services (stateless)
data_engine = DataEngine()
rime_api_key = os.getenv("RIME_API_KEY", "")
gemini_api_key = os.getenv("GEMINI_API_KEY", "")

@app.on_event("startup")
async def verify_rime_catalog():
    """
    Rime Preflight Check:
    Validates our configured model and voice against the live catalog.
    Rules require using the current catalog rather than a stale speaker list.
    """
    if not rime_api_key:
        return
        
    try:
        import httpx
        async with httpx.AsyncClient() as client:
            resp = await client.get("https://users.rime.ai/data/voices/all-v2.json", timeout=10.0)
            if resp.status_code == 200:
                catalog = resp.json()
                model = RimeTTS.MODEL_ID
                speaker = RimeTTS.SPEAKER
                
                # The catalog structure is usually { model: { language: [speakers] } }
                # We want to ensure 'coda' is supported and 'celeste' is in it.
                if model in catalog:
                    # Search across all languages for the speaker
                    found = any(speaker in (speakers if isinstance(speakers, list) else []) for speakers in catalog[model].values())
                    # Alternatively, if it's a simple list in 'eng'
                    is_in_eng = "eng" in catalog[model] and isinstance(catalog[model]["eng"], list) and speaker in catalog[model]["eng"]
                    
                    if is_in_eng or found:
                        logger.info(f"✅ Rime Preflight Check Passed: Model '{model}' and Voice '{speaker}' are active in the live catalog.")
                    else:
                        logger.warning(f"⚠️ Rime Preflight Warning: Voice '{speaker}' not found for model '{model}' in live catalog.")
                else:
                    logger.warning(f"⚠️ Rime Preflight Warning: Model '{model}' not found in live catalog.")
    except Exception as e:
        logger.error(f"Failed to run Rime catalog preflight check: {e}")

class MetricsCollector:
    def __init__(self):
        self.queries = deque(maxlen=100)
        self.total_queries = 0

    def add_metric(self, metric: dict):
        self.queries.append(metric)
        self.total_queries += 1

    def get_metrics(self):
        queries_list = list(self.queries)
        if not queries_list:
            return {
                "queries": [],
                "averages": {"avg_filler": 0, "avg_llm": 0, "avg_tts": 0, "avg_total": 0},
                "query_count": self.total_queries
            }
        
        return {
            "queries": queries_list,
            "averages": {
                "avg_filler": sum(m.get("filler_latency_ms", 0) for m in queries_list) / len(queries_list),
                "avg_llm": sum(m.get("llm_latency_ms", 0) for m in queries_list) / len(queries_list),
                "avg_tts": sum(m.get("tts_latency_ms", 0) for m in queries_list) / len(queries_list),
                "avg_total": sum(m.get("total_latency_ms", 0) for m in queries_list) / len(queries_list)
            },
            "query_count": self.total_queries
        }

metrics_collector = MetricsCollector()



@app.get("/health")
async def health_check():
    """Health check endpoint."""
    return {
        "status": "healthy",
        "rime_configured": bool(rime_api_key),
        "gemini_configured": bool(gemini_api_key),
    }


@app.get("/debug-llm")
async def debug_llm():
    """Debug endpoint to test LLM directly."""
    try:
        llm = LLMService(api_key=gemini_api_key)
        datasets = data_engine.list_datasets()
        result = await llm.analyze_query("Show me total sales by region", [], datasets)
        return {"status": "ok", "result": result}
    except Exception as e:
        import traceback
        return {"status": "error", "error": str(e), "traceback": traceback.format_exc()}


@app.get("/api/datasets")
async def list_datasets():
    """List available datasets with metadata."""
    return {"datasets": data_engine.list_datasets()}


@app.post("/api/upload-csv")
async def upload_csv(file: UploadFile = File(...)):
    """Upload a CSV dataset."""
    if not file.filename.endswith('.csv'):
        raise HTTPException(status_code=400, detail="Only CSV files are supported")
    
    # Read file content and check size (10MB limit)
    contents = await file.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="File too large (max 10MB)")
        
    try:
        # Parse CSV
        df = pd.read_csv(io.BytesIO(contents))
        
        # Auto-generate dataset ID from filename (lowercase, no spaces)
        dataset_id = file.filename.rsplit('.', 1)[0].lower().replace(" ", "_")
        
        # Store in data_engine
        data_engine.add_dataset(dataset_id, df, f"Uploaded dataset: {file.filename}")
        
        return {
            "id": dataset_id,
            "name": dataset_id,
            "columns": list(df.columns),
            "row_count": len(df),
            "sample_data": df.head(5).to_dict(orient="records")
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Failed to parse CSV: {str(e)}")


@app.get("/api/metrics")
async def get_metrics():
    """Get query latency metrics."""
    return metrics_collector.get_metrics()



@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    """
    Main WebSocket endpoint for voice-driven data analysis.
    Each connection gets its own ConversationState, RimeTTS, and LLMService.
    """
    await websocket.accept()
    logger.info("WebSocket client connected")

    # Per-connection state
    conv_state = ConversationState()
    tts_service = RimeTTS(api_key=rime_api_key)
    llm_service = LLMService(api_key=gemini_api_key)
    active_query_tasks: dict[int, asyncio.Task] = {}

    async def send_json_safe(data: dict):
        """Send JSON to client, handling closed connections."""
        try:
            await websocket.send_json(data)
        except Exception:
            pass

    async def handle_query(text: str, gen_id: int):
        """
        Full query pipeline:
        1. Send filler speech immediately
        2. Run LLM analysis
        3. Execute data query
        4. Stream Rime TTS audio
        5. Send chart data
        """
        cancel_event = conv_state.active_tasks.get(gen_id)
        if not cancel_event:
            return

        t_start = time.monotonic()
        logger.info(f"[gen={gen_id}] Processing query: {text[:80]}...")

        try:
            # Get conversation context and available datasets
            context_data = conv_state.get_context_for_llm()
            datasets = data_engine.list_datasets()

            # --- STEP 1: Fire filler speech ASAP (< 500ms target) ---
            filler_task = asyncio.create_task(
                tts_service.synthesize_filler(text)
            )
            # --- STEP 2: Fire LLM analysis concurrently ---
            llm_task = asyncio.create_task(
                llm_service.analyze_query(text, context_data, datasets)
            )

            # Wait for filler (should be fast for short phrases)
            filler_audio = await filler_task
            t_filler = time.monotonic()
            filler_latency_ms = (t_filler - t_start) * 1000
            logger.info(f"[gen={gen_id}] Filler latency: {filler_latency_ms:.0f}ms")

            if not cancel_event.is_set() and filler_audio:
                # Send filler transcript
                await send_json_safe({
                    "type": "transcript",
                    "text": tts_service.last_filler_text,
                    "generationId": gen_id,
                    "isFiller": True
                })
                # Send filler audio
                await send_json_safe({
                    "type": "audio",
                    "data": filler_audio,
                    "generationId": gen_id,
                    "isFinal": False
                })
                await send_json_safe({
                    "type": "status",
                    "state": "speaking",
                    "generationId": gen_id
                })

            # --- STEP 3: Wait for LLM result ---
            llm_result = await llm_task
            if cancel_event.is_set():
                logger.info(f"[gen={gen_id}] Cancelled after LLM")
                return
            
            conv_state.store_query_plan(llm_result)

            t_llm = time.monotonic()
            logger.info(f"[gen={gen_id}] LLM latency: {(t_llm - t_start) * 1000:.0f}ms")

            # --- STEP 4: Execute data query ---
            query_result = await data_engine.execute_query(llm_result, cancel_event)
            if cancel_event.is_set():
                logger.info(f"[gen={gen_id}] Cancelled after data query")
                return

            # --- STEP 5: Send chart data ---
            if "data" in query_result and query_result["data"]:
                chart_config = llm_result.get("chart_config") or {}
                chart_payload = {
                    "type": "chart",
                    "chartType": llm_result.get("chart_type", "bar"),
                    "responseType": llm_result.get("response_type", "chart_and_insight"),
                    "insights": llm_result.get("detailed_insights", ""),
                    "tableData": query_result.get("data", []),
                    "tableColumns": query_result.get("columns", []),
                    "data": query_result["data"],
                    "title": chart_config.get("title", "Analysis Result"),
                    "xKey": chart_config.get("x", ""),
                    "yKey": chart_config.get("y", ""),
                    "generationId": gen_id
                }
                await send_json_safe(chart_payload)

            insights = llm_result.get("detailed_insights", "")
            if insights:
                await send_json_safe({
                    "type": "transcript",
                    "text": insights,
                    "generationId": gen_id,
                    "isFiller": False,
                    "isInsight": True
                })

            # --- STEP 6: Stream main spoken response via Rime TTS ---
            spoken_text = llm_result.get("spoken_response", "Here are the results.")
            if cancel_event.is_set():
                return

            await send_json_safe({
                "type": "transcript",
                "text": spoken_text,
                "generationId": gen_id,
                "isFiller": False
            })
            await send_json_safe({
                "type": "status",
                "state": "speaking",
                "generationId": gen_id
            })

            # Stream audio chunks
            chunk_count = 0
            async for chunk_b64 in tts_service.synthesize_streaming(spoken_text, gen_id):
                if cancel_event.is_set():
                    logger.info(f"[gen={gen_id}] Cancelled during TTS streaming (after {chunk_count} chunks)")
                    return
                await send_json_safe({
                    "type": "audio",
                    "data": chunk_b64,
                    "generationId": gen_id,
                    "isFinal": False
                })
                chunk_count += 1

            # Send final audio marker
            if not cancel_event.is_set():
                await send_json_safe({
                    "type": "audio",
                    "data": "",
                    "generationId": gen_id,
                    "isFinal": True
                })
                conv_state.add_response(spoken_text, gen_id, was_heard=True)
                conv_state.mark_heard(gen_id)
                await send_json_safe({
                    "type": "status",
                    "state": "idle",
                    "generationId": gen_id
                })

                t_end = time.monotonic()
                total_latency_ms = (t_end - t_start) * 1000
                llm_latency_ms = (t_llm - t_start) * 1000
                tts_latency_ms = total_latency_ms - llm_latency_ms # Approx TTS processing/streaming latency
                
                logger.info(
                    f"[gen={gen_id}] Complete. Total: {total_latency_ms:.0f}ms, "
                    f"Chunks: {chunk_count}"
                )
                
                # Record metrics
                metrics_collector.add_metric({
                    "timestamp": time.time(),
                    "query_text": text[:50],
                    "filler_latency_ms": filler_latency_ms,
                    "llm_latency_ms": llm_latency_ms,
                    "tts_latency_ms": tts_latency_ms,
                    "total_latency_ms": total_latency_ms
                })

        except asyncio.CancelledError:
            logger.info(f"[gen={gen_id}] Task cancelled")
        except Exception as e:
            logger.error(f"[gen={gen_id}] Error: {e}", exc_info=True)
            if not cancel_event.is_set():
                await send_json_safe({
                    "type": "error",
                    "message": f"Analysis failed: {str(e)}",
                    "generationId": gen_id
                })
                await send_json_safe({
                    "type": "status",
                    "state": "idle",
                    "generationId": gen_id
                })

    try:
        while True:
            raw = await websocket.receive_text()
            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                logger.warning("Invalid JSON received")
                continue

            msg_type = data.get("type")

            if msg_type == "query":
                text = data.get("text", "").strip()
                if not text:
                    continue

                # Create new generation
                gen_id = conv_state.new_query(text)

                # Send processing status
                await send_json_safe({
                    "type": "status",
                    "state": "processing",
                    "generationId": gen_id
                })

                # Cancel any previous running task
                for old_id, old_task in list(active_query_tasks.items()):
                    if not old_task.done():
                        conv_state.interrupt(old_id)
                        tts_service.cancel()

                # Launch query handler
                task = asyncio.create_task(handle_query(text, gen_id))
                active_query_tasks[gen_id] = task

                # Cleanup completed tasks
                done_ids = [gid for gid, t in active_query_tasks.items() if t.done()]
                for gid in done_ids:
                    del active_query_tasks[gid]

            elif msg_type == "interrupt":
                target_gen_id = data.get("generationId")
                if target_gen_id is not None:
                    # Cancel the specified generation
                    conv_state.interrupt(target_gen_id)
                    tts_service.cancel()

                    await send_json_safe({
                        "type": "interrupted",
                        "generationId": target_gen_id
                    })
                    await send_json_safe({
                        "type": "status",
                        "state": "idle",
                        "generationId": target_gen_id
                    })
                    logger.info(f"[gen={target_gen_id}] Interrupted by client")

    except WebSocketDisconnect:
        logger.info("WebSocket client disconnected")
        # Cancel all active tasks
        for task in active_query_tasks.values():
            if not task.done():
                task.cancel()
    except Exception as e:
        logger.error(f"WebSocket error: {e}", exc_info=True)


if __name__ == "__main__":
    import uvicorn
    port = int(os.getenv("PORT", "8000"))
    logger.info(f"Starting DataForge on port {port}")
    uvicorn.run(app, host="0.0.0.0", port=port)
